In [5]:
import pandas as pd
from pathlib import Path
DATA = Path("../../data")
pd.set_option("display.max_columns", None)

b18 = pd.read_csv(DATA / "derived_oh15_2018_blocks.csv", dtype={"GEOID20":str, "cty3":str})
b22 = pd.read_csv(DATA / "derived_oh15_2022_blocks.csv", dtype={"GEOID20":str, "cty3":str})

# Merge on block GEOID (identical block set both years)
blocks = b18.merge(
    b22[["GEOID20","precinct_2022","dem_2022","rep_2022","tot_2022"]],
    on="GEOID20", how="outer")
for c in ["dem_2018","rep_2018","oth_2018","tot_2018","dem_2022","rep_2022","tot_2022"]:
    blocks[c] = blocks[c].fillna(0)
print(f"Merged blocks: {len(blocks):,}")
print("2018 tot:", round(blocks["tot_2018"].sum()), " | 2022 tot:", round(blocks["tot_2022"].sum()))

Merged blocks: 16,658
2018 tot: 264876  | 2022 tot: 247173


In [6]:
oh_county_names = {'023':'Clark','047':'Fayette','049':'Franklin','071':'Highland',
                   '097':'Madison','109':'Miami','129':'Pickaway'}

county = blocks.groupby("cty3").agg(
    dem_2018=("dem_2018","sum"), rep_2018=("rep_2018","sum"), tot_2018=("tot_2018","sum"),
    dem_2022=("dem_2022","sum"), rep_2022=("rep_2022","sum"), tot_2022=("tot_2022","sum"),
).round(0).astype(int)
county.insert(0, "county", county.index.map(oh_county_names))
county["dshare_2018"] = (county["dem_2018"]/(county["dem_2018"]+county["rep_2018"])*100).round(1)
county["dshare_2022"] = (county["dem_2022"]/(county["dem_2022"]+county["rep_2022"])*100).round(1)

# add district total row
tot = county[["dem_2018","rep_2018","tot_2018","dem_2022","rep_2022","tot_2022"]].sum()
print("=== OH-15 baseline by county ===")
print(county.to_string(index=False))
print(f"\nDISTRICT: 2018 {int(tot['tot_2018']):,} votes ({county['dem_2018'].sum()/(county['dem_2018'].sum()+county['rep_2018'].sum())*100:.1f}% D) | "
      f"2022 {int(tot['tot_2022']):,} votes ({county['dem_2022'].sum()/(county['dem_2022'].sum()+county['rep_2022'].sum())*100:.1f}% D)")

=== OH-15 baseline by county ===
  county  dem_2018  rep_2018  tot_2018  dem_2022  rep_2022  tot_2022  dshare_2018  dshare_2022
   Clark      2932      6204      9136      2249      6632      8881         32.1         25.3
 Fayette      2069      6241      8466      1701      6733      8434         24.9         20.2
Franklin     98819     80907    181492     86803     76473    163275         55.0         53.2
Highland      2777     10585     13547      2474     11827     14301         20.8         17.3
 Madison      3534     10035     13819      3660     10371     14030         26.0         26.1
   Miami      4781     13777     18558      4184     14648     18832         25.8         22.2
Pickaway      4958     14482     19858      5022     14398     19420         25.5         25.9

DISTRICT: 2018 264,876 votes (45.7% D) | 2022 247,173 votes (42.9% D)


In [8]:
# Precinct-level baseline. Precinct identity differs by year, so we aggregate each year
# to its own precincts, then keep both as reference tables.

prec_2018 = (blocks.groupby(["cty3","precinct_2018"])
             .agg(dem=("dem_2018","sum"), rep=("rep_2018","sum"), tot=("tot_2018","sum"))
             .round(0).astype(int).reset_index())
prec_2018["dshare"] = (prec_2018["dem"]/(prec_2018["dem"]+prec_2018["rep"])*100).round(1)
prec_2018["county"] = prec_2018["cty3"].map(oh_county_names)
prec_2018["year"] = 2018

prec_2022 = (blocks.groupby(["cty3","precinct_2022"])
             .agg(dem=("dem_2022","sum"), rep=("rep_2022","sum"), tot=("tot_2022","sum"))
             .round(0).astype(int).reset_index())
prec_2022["dshare"] = (prec_2022["dem"]/(prec_2022["dem"]+prec_2022["rep"])*100).round(1)
prec_2022["county"] = prec_2022["cty3"].map(oh_county_names)
prec_2022["year"] = 2022

print(f"2018 precincts: {len(prec_2018)} | 2022 precincts: {len(prec_2022)}")
print("\nSample — highest-turnout 2022 precincts:")
print(prec_2022.rename(columns={"precinct_2022":"precinct"})
      .sort_values("tot", ascending=False)
      .head(10)[["county","precinct","dem","rep","tot","dshare"]].to_string(index=False))

2018 precincts: 525 | 2022 precincts: 506

Sample — highest-turnout 2022 precincts:
  county         precinct  dem  rep  tot  dshare
   Clark      PRECINCT MR  716 1808 2524    28.4
   Miami PRECINCT PIQUA 5  394 1291 1685    23.4
Pickaway           SCIOTO  401 1220 1621    24.7
Pickaway    CIRCLEVILLE 1  529 1059 1588    33.3
Pickaway         ASHVILLE  349 1071 1420    24.6
Pickaway       WASHINGTON  276 1070 1346    20.5
Pickaway           WALNUT  280 1065 1345    20.8
Franklin              AUD  712  565 1277    55.8
   Miami PRECINCT PIQUA 2  311  749 1060    29.3
   Miami PRECINCT PIQUA 4  302  714 1016    29.7


In [9]:
# Save the baseline tables
county.to_csv(DATA / "derived_oh15_baseline_county.csv", index=False)
prec_2018.rename(columns={"precinct_2018":"precinct"})[
    ["county","cty3","precinct","dem","rep","tot","dshare","year"]
].to_csv(DATA / "derived_oh15_baseline_precinct_2018.csv", index=False)
prec_2022.rename(columns={"precinct_2022":"precinct"})[
    ["county","cty3","precinct","dem","rep","tot","dshare","year"]
].to_csv(DATA / "derived_oh15_baseline_precinct_2022.csv", index=False)

print("Saved 3 baseline tables:")
print("  derived_oh15_baseline_county.csv        (7 counties)")
print(f"  derived_oh15_baseline_precinct_2018.csv ({len(prec_2018)} precincts)")
print(f"  derived_oh15_baseline_precinct_2022.csv ({len(prec_2022)} precincts)")

Saved 3 baseline tables:
  derived_oh15_baseline_county.csv        (7 counties)
  derived_oh15_baseline_precinct_2018.csv (525 precincts)
  derived_oh15_baseline_precinct_2022.csv (506 precincts)


In [11]:
# --- Current registered voters from VoteBuilder (2026 snapshot) ---
tables = pd.read_html(DATA / "Demographics_van_export.xls", encoding="utf-16")
van = tables[0].copy()
van.columns = ['precinct','reg_active','reg_inactive','dropped','applicant','unknown','total_people']
van = van[van['precinct'].astype(str).str.contains(r'\([0-9]', na=False)].reset_index(drop=True)
for c in ['reg_active','reg_inactive','dropped','applicant','unknown','total_people']:
    van[c] = pd.to_numeric(van[c], errors='coerce')

# Parse VAN code -> county FIPS + suffix -> join key matching RDH VTDST22
van['van_code']   = van['precinct'].str.extract(r'\(([^)]+)\)')
van['van_cty']    = van['van_code'].str.extract(r'^(\d+)').astype(int)
van['van_suffix'] = van['van_code'].str.extract(r'^\d+[- ]?([a-zA-Z]+)')
van_to_fips = {12:"023",24:"047",25:"049",36:"071",49:"097",55:"109",65:"129"}
van['cty3']     = van['van_cty'].map(van_to_fips)
van['join_key'] = van['cty3'] + van['van_suffix'].str.upper()

# Validate against known VAN totals
print("VAN precincts:", len(van))
print("Reg Active:", van['reg_active'].sum(), "(want 434,522)")
print("Total People:", van['total_people'].sum(), "(want 515,380)")

VAN precincts: 537
Reg Active: 434522 (want 434,522)
Total People: 515380 (want 515,380)


In [12]:
# Aggregate VAN registration to the RDH precinct code (join_key), summing any VAN splits
van_by_key = van.groupby('join_key').agg(
    reg_active=('reg_active','sum'),
    total_people=('total_people','sum')
).reset_index()

# We need the code on the 2022 vote side. Reload votes22's code<->name<->cty3 mapping.
import geopandas as gpd
v22 = gpd.read_file(DATA / "oh_gen_2022_prec/oh_2022_gen_prec_no_splits.shp")
v22['join_key'] = v22['VTDST22'].astype(str).str.upper()
v22['cty3'] = v22['COUNTYFP'].astype(str).str.zfill(3)
# map code -> the PRECINCT name your baseline table uses
code_to_name = v22[['join_key','cty3','PRECINCT']].drop_duplicates()

# Attach registration to the code->name map
reg_named = code_to_name.merge(van_by_key, on='join_key', how='left')
print("Precincts with registration attached:", reg_named['reg_active'].notna().sum(), "/", len(reg_named))

# District-level turnout rates (the headline numbers)
total_reg_active = van['reg_active'].sum()
total_reg_all = van['total_people'].sum()
print(f"\n=== DISTRICT TURNOUT RATES ===")
print(f"Registered (active):     {total_reg_active:,}")
print(f"Registered (all):        {total_reg_all:,}")
print(f"2018 turnout: {264876/total_reg_all*100:.1f}% of all reg | {264876/total_reg_active*100:.1f}% of active")
print(f"2022 turnout: {247173/total_reg_all*100:.1f}% of all reg | {247173/total_reg_active*100:.1f}% of active")
print(f"2026 proj (250K): {250000/total_reg_all*100:.1f}% of all reg | {250000/total_reg_active*100:.1f}% of active")

Precincts with registration attached: 523 / 8941

=== DISTRICT TURNOUT RATES ===
Registered (active):     434,522
Registered (all):        515,380
2018 turnout: 51.4% of all reg | 61.0% of active
2022 turnout: 48.0% of all reg | 56.9% of active
2026 proj (250K): 48.5% of all reg | 57.5% of active


In [13]:
# Save the district turnout summary
summary = pd.DataFrame({
    'year': ['2018', '2022', 'external_proj_ref'],
    'votes': [264876, 247173, 250000],
    'note': ['actual', 'actual', 'campaign projection (not ours) - shown for reference'],
    'rate_vs_all_reg': [51.4, 48.0, 48.5],
    'rate_vs_active': [61.0, 56.9, 57.5]
})
summary.to_csv(DATA / "derived_oh15_turnout_rates.csv", index=False)
print("Saved turnout rate summary.")
print(f"Registered: {total_reg_all:,} total / {total_reg_active:,} active")

Saved turnout rate summary.
Registered: 515,380 total / 434,522 active
